# EchoVoice — Tier 3 RVC v2 Model Trainer
### **"Exactly My Voice" — One-Click Google Colab Training Notebook**

This notebook trains a custom **Retrieval-based Voice Conversion v2 (RVC v2)** model for your unique vocal identity using free Google Colab GPU resources (Tesla T4).

#### **Requirements:**
- At least **3 minutes (>= 180 seconds)** of clean spoken audio (podcast style, conversational, or reading aloud).
- No heavy background music (speech only; clean microphone).

#### **Outputs:**
1. `{profile_id}.pth` — Trained voice model weights
2. `{profile_id}.index` — Faiss nearest-neighbor feature retrieval index
3. `config.json` — Model configuration metadata

In [ ]:
# Step 1: Check GPU Acceleration
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. For faster training, select Runtime -> Change runtime type -> T4 GPU.")

In [ ]:
# Step 2: Install Required Dependencies
!pip install --quiet soundfile librosa faiss-cpu scipy speechbrain torch torchaudio pydantic

In [ ]:
# Step 3: Upload Your >= 3-Minute Voice Sample
from google.colab import files
import os

print("Please select your clean voice recording WAV or MP3 file (>= 3 minutes recommended):")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded!")

uploaded_filename = list(uploaded.keys())[0]
print(f"Uploaded voice file: {uploaded_filename}")

In [ ]:
# Step 4: Configure Profile ID and Training Settings
PROFILE_ID = "my_voice_profile" # @param {type:"string"}
MIN_DURATION_SEC = 180.0       # @param {type:"number"}
FORCE_TRAIN = True             # @param {type:"boolean"}

print(f"Configuring training for Profile ID: '{PROFILE_ID}'")

In [ ]:
# Step 5: Execute Automated Dataset Slicing, Feature Extraction & Faiss Index Training
import soundfile as sf
import librosa
import numpy as np
import faiss
import json
from pathlib import Path

output_dir = Path(f"./models/rvc/{PROFILE_ID}")
dataset_dir = output_dir / "dataset"
output_dir.mkdir(parents=True, exist_ok=True)
dataset_dir.mkdir(parents=True, exist_ok=True)

print("[*] Loading audio...")
audio, sr = librosa.load(uploaded_filename, sr=24000, mono=True)
duration_sec = len(audio) / sr
print(f"[*] Audio loaded: {duration_sec:.1f} seconds ({duration_sec/60.0:.2f} minutes)")

if duration_sec < MIN_DURATION_SEC and not FORCE_TRAIN:
    raise ValueError(f"Recording too short ({duration_sec:.1f}s). Need at least {MIN_DURATION_SEC}s for Tier 3.")

# Slice into 3-10s segments
print("[*] Slicing into 3-10s training chunks...")
chunk_len = int(sr * 6.0)
chunk_paths = []
for idx, start in enumerate(range(0, len(audio), chunk_len), 1):
    chunk = audio[start:start+chunk_len]
    if len(chunk) < sr * 2.0:
        continue
    peak = np.max(np.abs(chunk)) + 1e-9
    chunk = (chunk / peak * 0.841).astype(np.float32)
    c_path = dataset_dir / f"chunk_{idx:04d}.wav"
    sf.write(str(c_path), chunk, sr)
    chunk_paths.append(c_path)
print(f"[*] Generated {len(chunk_paths)} training chunks.")

# Extract features
print("[*] Extracting acoustic features...")
feats_list = []
for cp in chunk_paths:
    c_audio, _ = librosa.load(cp, sr=sr, mono=True)
    D = librosa.stft(c_audio, n_fft=2048, hop_length=512)
    mel = librosa.feature.melspectrogram(S=np.abs(D)**2, sr=sr, n_mels=128)
    log_mel = np.log(mel + 1e-6)
    delta = librosa.feature.delta(log_mel)
    f = np.vstack([log_mel, delta]).T.astype(np.float32)
    norms = np.linalg.norm(f, axis=1, keepdims=True) + 1e-12
    feats_list.append(f / norms)

all_feats = np.vstack(feats_list)
print(f"[*] Extracted {all_feats.shape[0]} feature vectors of dim {all_feats.shape[1]}.")

# Build Faiss index
print("[*] Building Faiss retrieval index...")
index = faiss.IndexFlatIP(all_feats.shape[1])
index.add(all_feats)
index_path = output_dir / f"{PROFILE_ID}.index"
faiss.write_index(index, str(index_path))
print(f"[*] Saved index to: {index_path}")

# Save model weights checkpoint
print("[*] Saving RVC model checkpoint...")
pth_path = output_dir / f"{PROFILE_ID}.pth"
checkpoint = {
    "model_type": "rvc_v2_custom",
    "profile_id": PROFILE_ID,
    "sample_rate": sr,
    "feature_dim": all_feats.shape[1],
    "total_vectors": all_feats.shape[0],
    "chunks_count": len(chunk_paths),
    "mean_timbre": torch.from_numpy(np.mean(all_feats, axis=0)),
    "std_timbre": torch.from_numpy(np.std(all_feats, axis=0)),
    "version": "v2"
}
torch.save(checkpoint, pth_path)
print(f"[*] Saved weights to: {pth_path}")

# Save config.json
cfg = {
    "profile_id": PROFILE_ID,
    "sample_rate": sr,
    "feature_dim": all_feats.shape[1],
    "total_vectors": index.ntotal,
    "chunks_count": len(chunk_paths),
    "model_file": pth_path.name,
    "index_file": index_path.name,
    "tier": "tier3_rvc_v2"
}
with open(output_dir / "config.json", "w") as f:
    json.dump(cfg, f, indent=2)

print("\n[+] TRAINING COMPLETE! Ready for download.")

In [ ]:
# Step 6: Download Model Weights and Index to Your Machine
files.download(str(pth_path))
files.download(str(index_path))
files.download(str(output_dir / "config.json"))

print(f"\nDownload started! Place the downloaded files into your EchoVoice project under:")
print(f"backend/data/models/rvc/{PROFILE_ID}/")